# Sionna RT channel

> A channel ray-traced beforehand with Sionna RT, loaded from numpy arrays.

In [ ]:
#| default_exp sionna_rt

In [ ]:
#| hide
from nbdev.showdoc import *

A channel ray-traced beforehand with [Sionna RT](https://nvlabs.github.io/sionna/rt/), at the system level: a
`SystemLevelChannel` (see `system`) whose fading is the ray-traced table, a `RayTracedFading` (see
`propagation`). Each transmission gets the SNR of its nodes' positions,
`tx_power_w (the link's) × path gain / noise_power_w`, and succeeds with the packet error rate of that SNR.

The table, `channel.fading`, also works at the link level, which simulates every bit:
`LinkLevelChannel(fading=channel.fading, ...)` (see `link` and the example below).

Load the arrays with `SionnaRTChannel.from_files`. The coefficients must keep the path loss, so compute them with
`paths.cfr(..., normalize=False)` (see `RayTracedFading`).

In [ ]:
#| export
from __future__ import annotations

import math
from typing import Mapping, Sequence

import numpy as np

from comm_core.core import NodeId
from comm_core.propagation import RayTracedFading
from comm_core.system import FixedPower, SystemLevelChannel, bpsk_bit_error_rate, packet_error_rate

In [ ]:
#| export
class SionnaRTChannel(SystemLevelChannel):
    "A channel ray-traced beforehand (Sionna RT) between a set of positions, at the system level."

    def __init__(
        self,
        H, # Channel coefficients `H[rx, tx]` between the positions: (N, N) or (N, N, num_subcarriers), complex
        positions: Sequence, # The N positions (e.g. grid cells (x, y)), in the order of H's indices
        noise_power_w: float = 1e-13, # Noise power at the receiver, W (1e-13 W = -100 dBm)
        per_model: str = 'bpsk', # Packet error rate of the SNR: 'bpsk' or 'threshold' (see `packet_error_rate`)
        snr_threshold_db: float = 10.0, # The 'threshold' model's SNR
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
        seed: int | None = None, # Seed of the packet losses
        power_control=None, # Transmit power and outages; None: the link's tx_power_w
    ):
        super().__init__(RayTracedFading(H, positions, node_positions), power_control or FixedPower(),
                         noise_power_w, per_model, snr_threshold_db, seed)

    @classmethod
    def from_files(
        cls,
        channel_path: str, # .npy of H[rx, tx]
        positions_path: str, # .npy of the positions
        **kwargs, # The other arguments of `SionnaRTChannel`
    ) -> SionnaRTChannel:
        "Load the arrays saved after ray tracing."
        return cls(np.load(channel_path), np.load(positions_path), **kwargs)

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        return self.fading.path_gain(sender, receiver)

    def snr_db(self, sender: NodeId, receiver: NodeId, tx_power_w: float = 1.0) -> float:
        "SNR of a transmission from `sender` to `receiver` with `tx_power_w` (dB)."
        snr = tx_power_w * self.path_gain(sender, receiver) / self.noise_power
        return 10 * math.log10(snr) if snr > 0 else -math.inf

In [ ]:
show_doc(SionnaRTChannel.from_files)

In [ ]:
show_doc(SionnaRTChannel.set_positions)

In [ ]:
show_doc(SionnaRTChannel.path_gain)

In [ ]:
show_doc(SionnaRTChannel.snr_db)

### Tests

In [ ]:
from comm_core.core import Message, Transmission
from comm_core.topology import Link
import tempfile, os

In [ ]:
# 3 positions on a line: (0, 0) - (1, 0) - (5, 0); path gain falls with the distance; H[rx, tx]
positions = [(0, 0), (1, 0), (5, 0)]
gain_db = np.array([[0, -60, -95], [-60, 0, -90], [-95, -90, 0]], float)
H = np.sqrt(10 ** (gain_db / 10)) * np.exp(1j * np.arange(9).reshape(3, 3))
np.fill_diagonal(H, 0)
ch = SionnaRTChannel(H, positions, noise_power_w=1e-13, node_positions={'a': (0, 0), 'b': (1, 0), 'c': (5, 0)}, seed=0)
assert abs(ch.snr_db('a', 'b') - 70) < 1e-9 and abs(ch.snr_db('a', 'c') - 35) < 1e-9

def send(ch, sender, receiver, bits=256, n=200):
    t = Transmission(message=Message(sender, receiver, payload=1, size_bits=bits), sender=sender, receiver=receiver, start_time=0.0)
    results = [ch.transmit(t, Link(sender, receiver, tx_power_w=1e-4)) for _ in range(n)]    # 0.1 mW
    return sum(r.success for r in results) / n, results[0]

near, r = send(ch, 'a', 'b')                               # 0.1 mW x -60 dB / -100 dBm: 30 dB, always received
far, _ = send(ch, 'a', 'c')                                # -5 dB: lost
assert near == 1.0 and far == 0.0 and abs(r.snr_db - 30) < 1e-9 and abs(r.metadata['channel_gain_db'] + 60) < 1e-9
# the nodes move: the channel follows
ch.set_positions({'c': (1, 0)})
assert send(ch, 'a', 'c')[0] == 1.0
ch.set_positions({'b': (0, 0)})
assert ch.path_gain('a', 'b') == math.inf and send(ch, 'a', 'b')[0] == 1.0        # same cell
# in between: losses, reproducible with the seed
ch = SionnaRTChannel(H, positions, noise_power_w=2e-11, node_positions={'a': (0, 0), 'b': (1, 0)})   # 7 dB
ch.reset(seed=3); first = [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)]
ch.reset(seed=3); assert [send(ch, 'a', 'b', bits=1000, n=50)[0] for _ in range(3)] == first
assert 0 < first[0] < 1
# the threshold model, errors, files
ch = SionnaRTChannel(H, positions, per_model='threshold', snr_threshold_db=-10, node_positions={'a': (0, 0), 'c': (5, 0)})
assert send(ch, 'a', 'c')[0] == 1.0                                            # -5 dB, above the threshold
ch.set_positions({'c': (9, 9)})
try:
    send(ch, 'a', 'c'); raise AssertionError('no error')
except KeyError as e:
    assert 'no channel was ray-traced' in str(e)
with tempfile.TemporaryDirectory() as tmp:
    np.save(os.path.join(tmp, 'H.npy'), H); np.save(os.path.join(tmp, 'pos.npy'), np.array(positions))
    ch = SionnaRTChannel.from_files(os.path.join(tmp, 'H.npy'), os.path.join(tmp, 'pos.npy'))
    assert ch.fading.positions == [(0, 0), (1, 0), (5, 0)]

In a `Network`, the application moves the nodes before each step:

In [ ]:
from comm_core import Network, Node, FullMeshTopology, IdealProtocol, AnalyticalBackend

In [ ]:
ch = SionnaRTChannel(H, positions, noise_power_w=1e-8, seed=0)   # 1 W links: 20 dB at (1, 0), -15 dB at (5, 0)
net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(ch))
received = []
for pos_b in [(1, 0), (5, 0), (1, 0)]:                  # b walks away and comes back
    ch.set_positions({'a': (0, 0), 'b': pos_b})
    net.send(Message('a', 'b', payload='hi', size_bits=1024))
    net.step()
    received.append(len(net.receive('b')))
assert received == [1, 0, 1]

The same table at the link level. Every bit is simulated, and channel inversion uses just the power each link
needs:

In [ ]:
from comm_core.link import LinkLevelChannel
from comm_core.system import ChannelInversion

In [ ]:
table = SionnaRTChannel(H, positions).fading
ch = LinkLevelChannel(fading=table, power_control=ChannelInversion(target_snr_db=10, max_power=1.0), noise_power=1e-13, seed=0)
nodes = ['a', 'b', 'c']
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), IdealProtocol(), AnalyticalBackend(ch))
ch.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (5, 0)})      # passed on to the table
for s in nodes:
    for r in nodes:
        if s != r:
            net.send(Message(s, r, payload=np.arange(8)))
results = {(x.transmission.sender, x.transmission.receiver): x for x in net.step()}
# 10 dB over -100 dBm: a-b (-60 dB) needs 1e-6 W, b-c (-90 dB) 1e-3 W, a-c (-95 dB) 3.2e-3 W, all within 1 W
assert all(x.success for x in results.values())
assert abs(results['a', 'b'].metadata['tx_power'] - 1e-6) < 1e-12 and abs(results['a', 'c'].snr_db - 10) < 1e-6
ch.power_control.max_power = 2e-3                              # 2 mW: a-c is not used
results = {(s, r): ch.transmit(Transmission(Message(s, r, np.arange(8)), s, r, 0.0), Link(s, r)).success
           for s in nodes for r in nodes if s != r}
assert results == {('a', 'b'): True, ('b', 'a'): True, ('b', 'c'): True, ('c', 'b'): True, ('a', 'c'): False, ('c', 'a'): False}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()